## Statistical Tests

The following section contains various Statistical Tests (p-value tests). It aims to provide more insights and basis of comparison for the various models against each other.

The following tests are present:

- Cochran's Q test for **comparing all 5 models at once** across one overall test to determine whether there is a statistically significant difference in **all models'** performance
- McNemar's test for comparing **two models at a time** as an extension to determine statistically significant difference in performance **between 2 models** on the same test


### Conversion of predictions to correctness

Both tests are run on correctness (whether each prediction was correct). Hence, we create both an array of correct predictions and also create a correctness table, where for each row a 1 indicates a correct prediction and a 0 indicates an incorrect prediction.

In [ ]:
import pandas as pd
import numpy as np

y_true = np.array(y_test)
correct_df = pd.DataFrame({
    name: (np.array(pred) == y_true).astype(int)
    for name, pred in predictions.items()
})
print("Correctness table for first test sample:")
correct_df.head()

Correctness table for first test sample:


,NB,LR,SVM,RF,BiLSTM
0,0,0,1,0,1
1,1,1,1,1,1
2,1,1,1,1,1
3,1,1,1,1,1
4,1,1,1,1,1


### Cochran's Q test

Cochran’s Q test is used to determine whether there is an overall statistically significant difference in prediction performance across all five models on the same test set.

Null hypothesis ($H_0$): All five models have the same prediction performance on the same test set.

Alternative hypothesis ($H_1$): At least one model has a different prediction performance from the others on the same test set.

In [ ]:
from statsmodels.stats.contingency_tables import cochrans_q

q_result = cochrans_q(correct_df)
print("Cochran's Q statistic:", q_result.statistic)
print("p-value:", q_result.pvalue)

Cochran's Q statistic: 1334.8773200543233
p-value: 9.122933333614289e-288


**Observations on Cochran's Q test:**

With the extremely small p-value obtained, we reject the null hypothesis. We can conclude that there is a statistically significant difference in performance among the 5 models on the same test set. 

In order to determine where this difference lies, we shall perform McNemar's test next.

### McNemar's test

McNemar’s test is used to compare pairs of models on the same test set and determine whether their differences in prediction performance were statistically significant. Due to the large number of pair-wise comparisons, Bonferroni correction was applied to adjust for these multiple pair-wise comparisons and reduce the risk of false positives.

For each pair-wise comparison, we have the following hypotheses:

Null hypothesis (**$H_0$**): The two models have the same prediction performance on the same test set.

Alternative hypothesis (**$H_1$**): The two models have different prediction performance on the same test set.

In [ ]:
from statsmodels.stats.contingency_tables import mcnemar
from statsmodels.stats.multitest import multipletests
from itertools import combinations

def mcnemar_pair(y_true, pred1, pred2):
    pred1 = np.array(pred1)
    pred2 = np.array(pred2)
    y_true = np.array(y_true)

    correct1 = pred1 == y_true
    correct2 = pred2 == y_true

    b = np.sum((correct1 == 1) & (correct2 == 0))
    c = np.sum((correct1 == 0) & (correct2 == 1))

    table = [[0, b],
             [c, 0]]

    result = mcnemar(table, exact=True)
    return result.pvalue, b, c

pairwise_results = []
for m1, m2 in combinations(predictions.keys(), 2):
    p, b, c = mcnemar_pair(y_true, predictions[m1], predictions[m2])
    pairwise_results.append({
        "Model_1": m1,
        "Model_2": m2,
        "b": b,
        "c": c,
        "p_value": p
    })

pairwise_df = pd.DataFrame(pairwise_results)
pairwise_df["p_bonferroni"] = multipletests(pairwise_df["p_value"], method="bonferroni")[1]
pairwise_df["significant_0.05"] = pairwise_df["p_bonferroni"] < 0.05

pairwise_df = pairwise_df.sort_values(["Model_1", "Model_2"]).reset_index(drop=True)

models_list = list(predictions.keys())
p_matrix = pd.DataFrame("-", index=models_list, columns=models_list)

for _, row in pairwise_df.iterrows():
    m1, m2 = row["Model_1"], row["Model_2"]
    pval = row["p_bonferroni"]
    p_matrix.loc[m1, m2] = f"{pval:.4g}"
    p_matrix.loc[m2, m1] = f"{pval:.4g}"

print(p_matrix)

                NB         LR         SVM          RF      BiLSTM
NB               -  2.75e-161  5.445e-143  1.427e-127  6.023e-126
LR       2.75e-161          -           1       0.862      0.2242
SVM     5.445e-143          1           -           1      0.5968
RF      1.427e-127      0.862           1           -           1
BiLSTM  6.023e-126     0.2242      0.5968           1           -


**Observations from McNemar's test**

After Bonferroni correction, McNemar's test shows that only Naive Bayes remains significantly worse than all other models as all the p-values from its pair-wise comparisons are extremely small. On the other hand, the differences among Logistic Regression, SVM, Random Forest, and BiLSTM are not statistically significant as the p-wavlues are >0.05 at the 5% significance level, suggesting that these stronger models perform comparably on the test set.

---

### Statistical tests summary

From both tests, it suggests that Naive Bayes performs clearly worse, whereas the stronger models perform at a broadly comparable level. This is likely because Naive Bayes relies heavily on independent word frequency assumptions, so it performs well only when classes contain strong and explicit lexical cues. In contrast, the other 4 models are all better able to learn more flexible patterns or contextual information, which helps them handle harder and more ambiguous cases. 

Once again, since all of the stronger models appear to have relatively comparable performance, it suggests the same underlying problem of the task where difficulty appears to stem from class ambiguity in the data and complexity of the task, rather than from model choice alone.